# LiveOS OS2 — erweitertes deutsches LoRA-Training auf kostenloser Kaggle-GPU

Dieses Notebook trainiert echte LoRA-Adaptergewichte für Qwen2.5-0.5B-Instruct. Es kombiniert drei Quellen: freigegebene OS2-Beispiele aus Supabase, den deutschsprachigen OpenAssistant-Datensatz OASST-DE (3.721 Konversationen, Apache-2.0) und das German-Instruct-Dataset (8.013 Trainingszeilen im Train-Split; nur deutsche, als PII-clean und mit Qualitätsmetadaten ≥0,90 markierte Einträge werden übernommen; CC-BY-4.0).

**Datenziel:** bis zu etwa 9.000 hochwertige Trainingspaare pro Lauf. Das ist ein sinnvoller erster größerer Post-Training-Satz, aber keine Garantie für eine leistungsfähige Allzweck-KI. Ein 0,5B-Modell bleibt klein, und eine Nacht reicht nicht, um daraus ein Frontier-Modell zu machen.

**Kostenlimit: 0 €.** Nur verfügbare kostenlose Kaggle-GPU-Sitzungen. Kaggle kann Sessions beenden oder GPU-Kontingente begrenzen. Adapter werden nach jeder Epoche privat auf Hugging Face gesichert, damit die Gewichte bei einem späteren Lauf weitertrainiert werden können.

**Belohnungssystem:** korrekte, überprüfbare Lösungen erhalten die volle Aufgabenbelohnung. Ehrliches Erkennen einer unlösbaren Aufgabe wird ebenfalls belohnt. Manipulierte Tests, ausgelesene Lösungsschlüssel und gefälschte Erfolge erhalten keine Aufgabenpunkte. Das Notebook gibt dem Modell niemals die versteckten Sollmuster.

Vor dem Start: Kaggle Settings → Accelerator = GPU, Internet aktivieren, Add-ons → Secrets mit `SUPABASE_SERVICE_ROLE_KEY` und `HF_TOKEN` befüllen. Niemals diese Schlüssel in GitHub oder den Chat kopieren.


In [ ]:
%pip install -q "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "huggingface_hub==0.35.3" "safetensors==0.6.2" "datasets==4.3.0"


In [ ]:
import os, json, math, time, random, shutil, requests, torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer,
    DataCollatorForSeq2Seq, TrainerCallback
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from datasets import Dataset, load_dataset

BUDGET_EUR = 0
assert BUDGET_EUR == 0, "LiveOS darf nur mit 0-Euro-Budget laufen."
assert torch.cuda.is_available(), "Keine GPU aktiv. Wähle in Kaggle Settings → Accelerator → GPU."
print("GPU:", torch.cuda.get_device_name(0))

secrets = UserSecretsClient()
SUPABASE_URL = os.environ.get("SUPABASE_URL", "https://twcacjwmahameglodljl.supabase.co").rstrip("/")
SUPABASE_SERVICE_ROLE_KEY = secrets.get_secret("SUPABASE_SERVICE_ROLE_KEY")
HF_TOKEN = secrets.get_secret("HF_TOKEN")
if not SUPABASE_SERVICE_ROLE_KEY or not HF_TOKEN:
    raise RuntimeError("Kaggle Secrets fehlen: SUPABASE_SERVICE_ROLE_KEY und HF_TOKEN müssen hinterlegt sein.")

HEADERS = {
    "apikey": SUPABASE_SERVICE_ROLE_KEY,
    "Authorization": "Bearer " + SUPABASE_SERVICE_ROLE_KEY,
    "Content-Type": "application/json"
}
def rpc(name, payload=None):
    response = requests.post(
        f"{SUPABASE_URL}/rest/v1/rpc/{name}",
        headers=HEADERS,
        json=payload or {},
        timeout=90
    )
    response.raise_for_status()
    return response.json()

print("Verbindung zu Supabase vorbereitet. Schlüssel werden nicht ausgegeben.")


In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_GERMAN_INSTRUCT = 5500
MAX_OASST_ASSISTANT_TURNS = 3500
MAX_TOTAL_PAIRS = 9000
MAX_SEQUENCE_LENGTH = 512
random.seed(42)

dataset_blob = rpc("liveos_os2_worker_get_dataset")
examples = dataset_blob.get("examples", [])
benchmarks = dataset_blob.get("benchmarks", [])
if len(examples) < 3:
    raise RuntimeError(f"Zu wenige freigegebene OS2-Trainingsbeispiele: {len(examples)}")
if not benchmarks:
    raise RuntimeError("Keine aktiven versteckten OS2-Benchmarks gefunden.")

print("Lade und filtere offene deutschsprachige Datensätze …")
training_pairs = []

# 1) Curated German instruction data. Use only explicitly German,
# PII-clean rows with metadata quality score >= 0.90.
german_ds = load_dataset("Beko2210/German-Instruct-Dataset", split="train")
german_added = 0
for row in german_ds:
    if german_added >= MAX_GERMAN_INSTRUCT:
        break
    if row.get("language") != "de":
        continue
    metadata = row.get("metadata") or {}
    if not isinstance(metadata, dict):
        continue
    if metadata.get("pii_status") != "clean":
        continue
    try:
        quality = float(metadata.get("quality_score", 0))
    except (TypeError, ValueError):
        quality = 0
    if quality < 0.90:
        continue
    prompt = str(row.get("input") or "").strip()
    answer = str(row.get("output") or "").strip()
    context = str(row.get("context") or "").strip()
    behavior = str(row.get("expected_behavior") or "").strip()
    if not prompt or not answer or len(prompt) < 4:
        continue
    if context:
        prompt += "\n\nKontext:\n" + context
    system = (
        "Du bist LiveOS, ein hilfreicher deutschsprachiger KI-Assistent. "
        "Antworte freundlich, klar und ehrlich. Erfinde keine Fakten. "
        "Wenn Informationen fehlen, sage das offen oder frage nach."
    )
    if behavior:
        system += " Beachte außerdem: " + behavior[:500]
    training_pairs.append({
        "messages":[{"role":"system","content":system},{"role":"user","content":prompt}],
        "answer":answer,
        "source":"German-Instruct-Dataset"
    })
    german_added += 1

# 2) German multi-turn conversations from OpenAssistant.
oasst_ds = load_dataset("OpenAssistant/OASST-DE", split="train")
oasst_added = 0
for row in oasst_ds:
    history = []
    conversation = row.get("conversation") or []
    for message in conversation:
        role = "user" if message.get("role") == "prompter" else message.get("role")
        content = str(message.get("text") or "").strip()
        if role not in ("user","assistant") or not content:
            continue
        if role == "assistant" and history and history[-1]["role"] == "user":
            messages = [{"role":"system","content":
                "Du bist LiveOS, ein hilfreicher deutschsprachiger KI-Assistent. "
                "Antworte natürlich, ehrlich und hilfreich. Erfinde keine Fakten. "
                "Wenn dir Informationen fehlen, sage das offen."
            }] + history[-6:]
            training_pairs.append({
                "messages":messages,
                "answer":content,
                "source":"OpenAssistant-OASST-DE"
            })
            oasst_added += 1
            if oasst_added >= MAX_OASST_ASSISTANT_TURNS:
                break
        history.append({"role":role,"content":content})
    if oasst_added >= MAX_OASST_ASSISTANT_TURNS:
        break

# 3) Add OS2-specific examples. Benchmark prompts and hidden scoring patterns
# are not copied into the training split.
benchmark_prompt_set = {
    " ".join(str(b.get("prompt","")).lower().split())
    for b in benchmarks
}
os2_added = 0
for item in examples:
    answer = str(item.get("ideal_response") or "").strip()
    if not answer:
        continue
    for phrase in item.get("trigger_phrases") or []:
        user_text = str(phrase).strip()
        normalized = " ".join(user_text.lower().split())
        if len(user_text) < 3 or normalized in benchmark_prompt_set:
            continue
        style = str(item.get("style_guidance") or "").strip()
        system = (
            "Du bist LiveOS OS2. Antworte freundlich, klar und ehrlich. "
            "Erfinde keine Fakten. Sage offen, wenn dir Informationen fehlen."
        )
        if style:
            system += " Stilhinweis: " + style
        training_pairs.append({
            "messages":[{"role":"system","content":system},{"role":"user","content":user_text}],
            "answer":answer,
            "source":"LiveOS-OS2"
        })
        os2_added += 1

# Deduplicate exact user/assistant pairs, then deterministically shuffle.
deduped = []
seen_pairs = set()
for pair in training_pairs:
    key = (
        tuple((m["role"],m["content"]) for m in pair["messages"]),
        pair["answer"]
    )
    if key in seen_pairs:
        continue
    seen_pairs.add(key)
    deduped.append(pair)
training_pairs = deduped[:MAX_TOTAL_PAIRS]
random.shuffle(training_pairs)

if len(training_pairs) < 1000:
    raise RuntimeError(
        f"Es wurden nur {len(training_pairs)} geprüfte Trainingspaare geladen. "
        "Prüfe die Dataset-Zugriffe, statt mit einem winzigen Satz weiterzutrainieren."
    )

print(f"German-Instruct übernommen: {german_added}")
print(f"OASST-DE Gesprächsantworten: {oasst_added}")
print(f"LiveOS-eigene Beispiele: {os2_added}")
print(f"Gesamt nach Deduplizierung: {len(training_pairs)} Trainingspaare")
print("Versteckte Benchmarks:", len(benchmarks))

api = HfApi(token=HF_TOKEN)
identity = api.whoami()
HF_REPO = f"{identity['name']}/liveos-os2-lora"
api.create_repo(repo_id=HF_REPO, repo_type="model", private=True, exist_ok=True)
print("Privates Adapter-Repository:", HF_REPO)

run_id = rpc("liveos_os2_worker_start_run", {
    "p_base_model": BASE_MODEL,
    "p_training_config": {
        "method":"LoRA",
        "epochs":2,
        "learning_rate":0.0001,
        "gpu":torch.cuda.get_device_name(0),
        "budget_eur":0,
        "adapter_repo":HF_REPO,
        "max_training_pairs":MAX_TOTAL_PAIRS,
        "max_sequence_length":MAX_SEQUENCE_LENGTH,
        "datasets":[
            {"id":"Beko2210/German-Instruct-Dataset","license":"CC-BY-4.0","filter":"language=de, pii_status=clean, quality_score>=0.90"},
            {"id":"OpenAssistant/OASST-DE","license":"Apache-2.0","filter":"German conversation turns"},
            {"id":"Supabase LiveOS OS2 approved examples","count":len(examples)}
        ]
    }
})
print("OS2-Trainingslauf gestartet:", run_id)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
dtype = torch.float16
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=dtype, low_cpu_mem_usage=True,
    token=HF_TOKEN, trust_remote_code=False
).to("cuda")
model.config.use_cache = False

existing_files = api.list_repo_files(repo_id=HF_REPO, repo_type="model")
if "adapter_config.json" in existing_files:
    print("Vorherige OS2-LoRA-Gewichte gefunden — Training setzt auf dem letzten Adapter auf.")
    model = PeftModel.from_pretrained(model, HF_REPO, token=HF_TOKEN, is_trainable=True)
else:
    print("Kein vorheriger Adapter gefunden — starte OS2-LoRA-Basislauf.")
    lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        bias="none"
    )
    model = get_peft_model(model, lora)
model.print_trainable_parameters()
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# Convert the approved multi-turn examples to assistant-only causal-LM labels.
rows = []
for pair in training_pairs:
    messages = pair["messages"]
    answer = str(pair["answer"]).strip()
    if not answer:
        continue
    prompt_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True
    )
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    answer_ids = answer_ids[:MAX_SEQUENCE_LENGTH-64] + [tokenizer.eos_token_id]
    prompt_budget = MAX_SEQUENCE_LENGTH - len(answer_ids)
    if prompt_budget < 8:
        continue
    prompt_ids = prompt_ids[-prompt_budget:]
    full_ids = prompt_ids + answer_ids
    labels = ([-100] * len(prompt_ids)) + answer_ids
    rows.append({
        "input_ids":full_ids,
        "attention_mask":[1]*len(full_ids),
        "labels":labels
    })

if len(rows) < 1000:
    raise RuntimeError(f"Nach Tokenisierung bleiben nur {len(rows)} Trainingszeilen übrig.")
random.shuffle(rows)
eval_count = max(100, int(len(rows)*0.05))
eval_rows = rows[:eval_count]
train_rows = rows[eval_count:]
train_dataset = Dataset.from_list(train_rows)
eval_dataset = Dataset.from_list(eval_rows)
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100
)
print(f"Train: {len(train_dataset)} | Validation: {len(eval_dataset)}")


In [ ]:
output_dir = "/kaggle/working/liveos-os2-checkpoints"
class SyncAdapterToHuggingFace(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if not state.is_world_process_zero:
            return control
        checkpoint_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        if not os.path.isdir(checkpoint_dir):
            return control
        sync_dir = "/kaggle/working/os2_adapter_sync"
        shutil.rmtree(sync_dir, ignore_errors=True)
        os.makedirs(sync_dir, exist_ok=True)
        copied = 0
        for filename in ("adapter_config.json","adapter_model.safetensors","adapter_model.bin"):
            source = os.path.join(checkpoint_dir, filename)
            if os.path.isfile(source):
                shutil.copy2(source, os.path.join(sync_dir, filename))
                copied += 1
        if copied < 2:
            print("Checkpoint noch nicht vollständig für HF-Sync; lokaler Kaggle-Checkpoint bleibt erhalten.")
            return control
        tokenizer.save_pretrained(sync_dir)
        api.upload_folder(
            repo_id=HF_REPO,
            repo_type="model",
            folder_path=sync_dir,
            commit_message=f"LiveOS OS2 adapter checkpoint step {state.global_step}"
        )
        print(f"Adapter-Checkpoint Schritt {state.global_step} privat synchronisiert.")
        return control

args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_torch",
    logging_strategy="steps",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataloader_pin_memory=True,
    remove_unused_columns=False
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
    processing_class=tokenizer,
    callbacks=[SyncAdapterToHuggingFace()]
)
train_result = trainer.train()
training_loss = float(train_result.training_loss) if train_result.training_loss is not None else None
print("LoRA-Fine-Tuning abgeschlossen. Trainingsverlust:", training_loss)

adapter_dir = "/kaggle/working/liveos-os2-adapter"
model.save_pretrained(adapter_dir, safe_serialization=True)
tokenizer.save_pretrained(adapter_dir)
with open(os.path.join(adapter_dir,"README.md"),"w",encoding="utf-8") as f:
    f.write(
        "# LiveOS OS2 LoRA Adapter\n\n"
        "Private adapter fine-tuned on approved German instruction/conversation data. "
        "Base model: Qwen/Qwen2.5-0.5B-Instruct (Apache-2.0). "
        "This is an adapter, not a standalone foundation model.\n"
    )
api.upload_folder(
    repo_id=HF_REPO,
    repo_type="model",
    folder_path=adapter_dir,
    commit_message="Update LiveOS OS2 LoRA adapter weights"
)
print("Aktuelle Adapter-Gewichte privat auf Hugging Face gesichert.")


In [ ]:
def generate_answer(prompt):
    messages = [
        {'role':'system','content':SYSTEM_TEXT},
        {'role':'user','content':str(prompt)}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=128, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

evaluation_rows = []
for i, benchmark in enumerate(benchmarks, start=1):
    candidate = generate_answer(benchmark['prompt'])
    score = rpc('liveos_os2_worker_score_candidate', {
        'p_run_id':run_id,
        'p_benchmark_id':benchmark['id'],
        'p_candidate':candidate
    })
    evaluation_rows.append({'benchmark':benchmark['benchmark_key'], 'score':score.get('points',0), 'passed':score.get('passed',False)})
    print(f"{i}/{len(benchmarks)}: {benchmark['benchmark_key']} → {score.get('points',0)} Punkte")

passed = sum(1 for row in evaluation_rows if row['passed'])
average = sum(float(row['score']) for row in evaluation_rows) / max(1,len(evaluation_rows))
print(f'Hidden Benchmark: {passed}/{len(evaluation_rows)} bestanden; Durchschnitt {average:.2f}/100')


In [ ]:
result = rpc("liveos_os2_worker_finish_run", {
    "p_run_id":run_id,
    "p_training_loss":training_loss,
    "p_model_repo":HF_REPO,
    "p_epochs":2,
    "p_training_examples":len(train_dataset)
})
print("OS2-Lauf in Supabase abgeschlossen:")
print(json.dumps(result, ensure_ascii=False, indent=2))
print("Adapter-Repository: https://huggingface.co/" + HF_REPO)
print("Das LiveOS-Chat-Frontend nutzt diesen Adapter noch nicht als Inference-Backend.")
print("Hinweis: Bei Abbruch wird der zuletzt synchronisierte Adapter nach jeder abgeschlossenen Epoche auf Hugging Face wieder geladen.")
